# 11_multi_container_reconciliation

Generated from `alteryx-workflows\11_multi_container_reconciliation.yxmd` by compiler 2.2.0.

## How to use this notebook

1. Review and complete the source configuration.
2. Run cells from top to bottom.
3. Inspect the output previews and validate semantic parity.

> **Important:** Generated transformations are unvalidated. Confirm schemas, row counts, business rules, and destinations before promotion.


## Runtime imports

Imports and runtime helpers required by the generated PySpark transformations.


In [ ]:
from pyspark.sql import Window, functions as F


## Source configuration

Replace each empty value with an accessible Fabric or OneLake path before running the ingestion cells.

| Tool ID | Generated DataFrame |
| --- | --- |
| `1` | `src_businessdate` |
| `2` | `src_settlement_current` |
| `3` | `src_alpha_current` |
| `4` | `src_alpha_previous` |
| `5` | `src_ref_details` |
| `6` | `src_beta_current` |
| `7` | `src_beta_previous` |
| `8` | `src_manifestfiles` |
| `9` | `src_settlement_previous` |


In [ ]:
source_paths = {
    "1": "",  # src_businessdate
    "2": "",  # src_settlement_current
    "3": "",  # src_alpha_current
    "4": "",  # src_alpha_previous
    "5": "",  # src_ref_details
    "6": "",  # src_beta_current
    "7": "",  # src_beta_previous
    "8": "",  # src_manifestfiles
    "9": "",  # src_settlement_previous
}


## Source ingestion

Loads and validates source datasets before transformation.


### Source ingestion 1 of 3

Produces `src_businessdate_Path`, `src_businessdate`, `src_settlement_current_Path`, `src_settlement_current`, and 4 more.


In [ ]:
src_businessdate_Path = source_paths.get('1')
if not src_businessdate_Path: raise ValueError("Configure source_paths['1'] before running")
src_businessdate = spark.read.option('header', True).option('inferSchema', True).csv(src_businessdate_Path)
src_settlement_current_Path = source_paths.get('2')
if not src_settlement_current_Path: raise ValueError("Configure source_paths['2'] before running")
src_settlement_current = spark.read.option('header', True).option('inferSchema', True).csv(src_settlement_current_Path)
src_alpha_current_Path = source_paths.get('3')
if not src_alpha_current_Path: raise ValueError("Configure source_paths['3'] before running")
src_alpha_current = spark.read.option('header', True).option('inferSchema', True).csv(src_alpha_current_Path)
src_alpha_previous_Path = source_paths.get('4')
if not src_alpha_previous_Path: raise ValueError("Configure source_paths['4'] before running")
src_alpha_previous = spark.read.option('header', True).option('inferSchema', True).csv(src_alpha_previous_Path)


### Source ingestion 2 of 3

Produces `src_ref_details_Path`, `src_ref_details`, `src_beta_current_Path`, `src_beta_current`, and 4 more.


In [ ]:
src_ref_details_Path = source_paths.get('5')
if not src_ref_details_Path: raise ValueError("Configure source_paths['5'] before running")
src_ref_details = spark.read.option('header', True).option('inferSchema', True).csv(src_ref_details_Path)
src_beta_current_Path = source_paths.get('6')
if not src_beta_current_Path: raise ValueError("Configure source_paths['6'] before running")
src_beta_current = spark.read.option('header', True).option('inferSchema', True).csv(src_beta_current_Path)
src_beta_previous_Path = source_paths.get('7')
if not src_beta_previous_Path: raise ValueError("Configure source_paths['7'] before running")
src_beta_previous = spark.read.option('header', True).option('inferSchema', True).csv(src_beta_previous_Path)
src_manifestfiles_Path = source_paths.get('8')
if not src_manifestfiles_Path: raise ValueError("Configure source_paths['8'] before running")
src_manifestfiles = spark.read.option('header', True).option('inferSchema', True).csv(src_manifestfiles_Path)


### Source ingestion 3 of 3

Produces `src_settlement_previous_Path`, `src_settlement_previous`.


In [ ]:
src_settlement_previous_Path = source_paths.get('9')
if not src_settlement_previous_Path: raise ValueError("Configure source_paths['9'] before running")
src_settlement_previous = spark.read.option('header', True).option('inferSchema', True).csv(src_settlement_previous_Path)


## Transformations

Applies the translated Alteryx operations in dependency order.


### Transformations 1 of 3

Produces `qalpha_current_calculated`, `qalpha_previous_calculated`, `qalpha_beta_current`, `qalpha_beta_previous`, and 4 more.


In [ ]:
qalpha_current_calculated = src_alpha_current.withColumn("CalculatedAmount", F.col("Amount").cast('double'))
qalpha_previous_calculated = src_alpha_previous.withColumn("CalculatedAmount", F.col("Amount").cast('double'))
qalpha_beta_current = src_beta_current
qalpha_beta_previous = src_beta_previous
qalpha_beta_combined = src_beta_current.unionByName(src_beta_previous, allowMissingColumns=True)
qref_entitylookup = src_ref_details
qalpha_beta_enriched = qalpha_beta_combined.join(src_ref_details, (qalpha_beta_combined["RecordKey"] == src_ref_details["RecordKey"]), 'inner').drop(src_ref_details["RecordKey"])
qref_accountlookup = src_ref_details


### Transformations 2 of 3

Produces `qalpha_beta_calculated`, `qsettlement_current_ready`, `qsettlement_current_calculated`, `qsettlement_current_matched`, and 4 more.


In [ ]:
qalpha_beta_calculated = qalpha_beta_enriched.join(src_ref_details, (qalpha_beta_enriched["RecordKey"] == src_ref_details["RecordKey"]), 'inner').drop(src_ref_details["RecordKey"])
qsettlement_current_ready = src_settlement_current
qsettlement_current_calculated = src_settlement_current.withColumn("CalculatedAmount", F.col("Amount").cast('double'))
qsettlement_current_matched = qsettlement_current_calculated.join(qalpha_current_calculated, (qsettlement_current_calculated["RecordKey"] == qalpha_current_calculated["RecordKey"]), 'inner').drop(qalpha_current_calculated["RecordKey"])
qsettlement_current_result = qsettlement_current_matched
qsettlement_previous_ready = src_settlement_previous
qsettlement_previous_calculated = src_settlement_previous.withColumn("CalculatedAmount", F.col("Amount").cast('double'))
qsettlement_previous_matched = qsettlement_previous_calculated.join(qalpha_previous_calculated, (qsettlement_previous_calculated["RecordKey"] == qalpha_previous_calculated["RecordKey"]), 'inner').drop(qalpha_previous_calculated["RecordKey"])


### Transformations 3 of 3

Produces `qsettlement_previous_result`, `qsettlement_combined`, `out_settlement`, `out_alpha_beta`.


In [ ]:
qsettlement_previous_result = qsettlement_previous_matched
qsettlement_combined = qsettlement_current_matched.unionByName(qsettlement_previous_matched, allowMissingColumns=True)
out_settlement = qsettlement_combined
out_alpha_beta = qalpha_beta_calculated


## Output preview

Displays terminal DataFrames for review. Replace preview-only output with approved destination writes after validation.


### Output preview 1 of 1


In [ ]:
display(src_businessdate)
display(src_manifestfiles)
display(qsettlement_combined)
display(qsettlement_combined)
display(qalpha_beta_calculated)
display(qalpha_beta_calculated)
